# Chains

## 传统 Chain 使用方式

### 基础链(废弃)

LLMChain 是 LangChain 早期的核心组件，用于将 LLM 和 Prompt 组合成一个可调用的链。

> **注意**: LLMChain 已被标记为遗留API，推荐使用 LCEL（`prompt | llm`）替代。

In [1]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# 1. 创建大模型实例
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 2. 创建 PromptTemplate
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请用一句话简单介绍{topic}是什么？",
)

# 3. 创建 LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# 4. 调用 chain
result = chain.invoke({"topic": "LangChain"})
print("LLMChain 输出:")
print(result["text"])

C:\Users\LING YONG\AppData\Local\Temp\ipykernel_10716\634874157.py:23: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


LLMChain 输出:
LangChain 是一个用于构建大语言模型（LLM）应用的开源框架，它提供了将语言模型与外部数据、工具和工作流连接起来的标准化接口，帮助开发者更便捷地创建智能应用。


### 顺序链
- SimpleSequentialChain,表示单个输入输出
- SequentialChian,表示多个输入输出

### 数学链
> LLMMathChain

### 路由链
> RouterChain

### 文档链
> StuffDocumentsChain

## LCEL 现代写法（推荐）

使用 `|` 管道符将 prompt 和 llm 组合成 chain，更简洁直观。

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# LCEL 写法：prompt | llm
prompt_lcel = ChatPromptTemplate.from_template("请用一句话简单介绍{topic}是什么？")
chain_lcel = prompt_lcel | llm

# 调用方式相同
result_lcel = chain_lcel.invoke({"topic": "LangChain"})
print("LCEL 输出:")
print(result_lcel.content)

LCEL 输出:
LangChain 是一个用于构建大语言模型（LLM）应用的开源框架，帮助开发者将大模型与外部数据、工具和工作流串联起来，从而快速搭建智能应用。


## 两种方式对比

| 特性 | LLMChain (传统) | LCEL (推荐) |
|------|----------------|-------------|
| 语法 | `LLMChain(llm=llm, prompt=prompt)` | `prompt \| llm` |
| 可读性 | 较低 | 高 |
| 扩展性 | 需要嵌套 Chain | 管道符组合 |
| 流式支持 | 有限 | 原生支持 |
| 状态 | 已废弃 | 当前推荐 |

## SequentialChain 示例

将多个 Chain 串联执行，前一个的输出作为后一个的输入。

## SimpleSequentialChain 示例

SimpleSequentialChain 是最简单的顺序链，每个步骤只有一个输入和一个输出：
- 输入 → Chain1 → 输出1 → Chain2 → 最终输出

In [ ]:
from langchain_classic.chains import SimpleSequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成标题
prompt_title = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}生成一个吸引人的标题。",
)
chain_title = LLMChain(llm=llm, prompt=prompt_title)

# 第二个链：根据标题写摘要
prompt_summary = PromptTemplate(
    input_variables=["title"],
    template="请根据标题'{title}'写一段50字以内的摘要。",
)
chain_summary = LLMChain(llm=llm, prompt=prompt_summary)

# 创建 SimpleSequentialChain（单输入单输出）
simple_chain = SimpleSequentialChain(
    chains=[chain_title, chain_summary],
    verbose=True,
)

# 执行：只需要传入最初的输入
result = simple_chain.invoke({"input": "Python编程入门"})
print("最终输出:")
print(result["output"])



> Entering new SimpleSequentialChain chain...
以下是一些吸引人的Python编程入门标题，供您选择：

---

## 🔥 热门风格

1. **《从零到一：开启你的Python编程之旅》**
2. **《Python零基础入门：人人都能学会的编程魔法》**
3. **《一行代码，无限可能——Python编程新手完全指南》**

## 🎯 直击痛点风格

4. **《告别编程恐惧症：30天轻松掌握Python》**
5. **《不会写代码？Python让你从小白变高手》**
6. **《为什么所有人都在学Python？这本书告诉你答案》**

## 🚀 趣味创意风格

7. **《和Python交个朋友：最友好的编程入门课》**
8. **《像搭积木一样学Python——趣味编程入门》**
9. **《人生苦短，我用Python——写给初学者的第一本书》**

## 💼 专业实用风格

10. **《Python编程基础与实战：从入门到上手》**
11. **《零基础Python开发入门与进阶》**

---

> 💡 **小建议**：选择标题时可以考虑您的**目标受众**——面向大众读者可以用更趣味化的标题，面向学生或职场人士则可以更突出实用性和学习效率。希望这些标题能给您灵感！😊
这份清单包含多种风格的Python入门标题，旨在通过不同角度吸引初学者，突显Python的易学性、实用性和趣味性。

> Finished chain.
最终输出:
这份清单包含多种风格的Python入门标题，旨在通过不同角度吸引初学者，突显Python的易学性、实用性和趣味性。


## SimpleSequentialChain vs SequentialChain

| 特性 | SimpleSequentialChain | SequentialChain |
|------|----------------------|-----------------|
| 输入/输出 | 每步单输入单输出 | 支持多输入多输出 |
| 参数 | `input` | `input_variables` |
| 返回值 | `output` | `output_variables` |
| 适用场景 | 简单串联 | 复杂数据流 |

In [5]:
from langchain_classic.chains import SequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成大纲
prompt_outline = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}写一个简短的大纲，包含3个要点。",
)
chain_outline = LLMChain(llm=llm, prompt=prompt_outline, output_key="outline")

# 第二个链：根据大纲写简介
prompt_intro = PromptTemplate(
    input_variables=["outline"],
    template="根据以下大纲写一段简短的介绍：\n{outline}",
)
chain_intro = LLMChain(llm=llm, prompt=prompt_intro, output_key="intro")

# 串联两个链
sequential_chain = SequentialChain(
    chains=[chain_outline, chain_intro],
    input_variables=["topic"],
    output_variables=["outline", "intro"],
    verbose=True,
)

# 执行
result = sequential_chain.invoke({"topic": "Python编程"})
print("大纲:")
print(result["outline"])
print("\n介绍:")
print(result["intro"])



> Entering new SequentialChain chain...

> Finished chain.
大纲:
一个适合初学者的Python编程简短大纲，包含三个核心要点：

1.  **基本语法与数据类型**
    *   学习变量、常见数据类型（如整数、字符串、列表、字典）。
    *   掌握基础运算符和控制流语句（如`if`条件判断、`for`/`while`循环）。

2.  **函数与模块化编程**
    *   学习如何定义和调用函数，理解参数和返回值。
    *   了解如何导入和使用标准库模块，以组织代码并复用现有功能。

3.  **面向对象与实战应用**
    *   初步理解类和对象的概念，了解如何使用`class`创建简单的对象。
    *   结合文件读写、网络请求等常用标准库，进行小型项目实践，如数据处理或自动化脚本。

这个大纲从基础语法出发，逐步构建到代码组织和实际应用，为你提供一条清晰的入门路径。

介绍:
欢迎来到 Python 编程的世界！这份精心设计的大专为你——零基础的初学者——铺设了一条清晰、循序渐进的学习路径。

我们将从**基石**开始：首先掌握变量、数字、文本和列表等基本“建材”，学会用条件判断和循环来指挥程序流程，这是编写任何程序的前提。

接着，我们将学习如何让代码变得**井然有序**。你将学会用函数封装常用操作，避免重复劳动，并探索强大的标准库模块，像使用现成的工具一样，直接调用别人写好的强大功能。

最终，我们将带你走向**实战应用**。你将初步接触面向对象的思想，理解如何用“类”来描述复杂事物，并综合运用文件操作、网络请求等技能，完成数据处理、自动化脚本等小型项目，真正感受编程解决实际问题的力量。

从基础语法到代码组织，再到动手实践，这是一条为你量身打造的、从入门到能做出有用东西的成长之路。让我们开始吧！


## LCEL 等效写法

使用 LCEL 的 `RunnablePassthrough` 和 `RunnableLambda` 实现同样的功能。

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# LCEL 写法：管道符串联
outline_prompt = ChatPromptTemplate.from_template("请为{topic}写一个简短的大纲，包含3个要点。")
intro_prompt = ChatPromptTemplate.from_template("根据以下大纲写一段简短的介绍：\n{outline}")

# 使用 LCEL 组合
chain_lcel = (
    {"topic": RunnablePassthrough()}
    | outline_prompt
    | llm
    | (lambda x: {"outline": x.content})
    | intro_prompt.partial(topic="Python编程")  # 这里简化处理
)

# 更清晰的 LCEL 写法
def extract_outline(response):
    return {"outline": response.content}

chain_lcel_clear = (
    outline_prompt | llm | extract_outline | intro_prompt | llm
)

result_lcel = chain_lcel_clear.invoke({"topic": "Python编程"})
print("LCEL SequentialChain 输出:")
print(result_lcel.content)